In [ ]:
import pandas as pd

df_train = pd.read_csv("../data/training_data.csv")
df_test = pd.read_csv("../data/testing_data.csv")

print("✅ Dataset chargé !")
print(df_train.head())

In [ ]:
from transformers import pipeline

# Modèle de sentiment arabe
sentiment_model = pipeline(
    "text-classification",
    model="CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment",
    top_k=None  # ← remplace return_all_scores=True
)

def get_sentiment_features(text):
    try:
        results = sentiment_model(text[:512])[0]
        scores = {r['label']: r['score'] for r in results}
        p_pos = scores.get('positive', 0.0)
        p_neg = scores.get('negative', 0.0)
        p_neu = scores.get('neutral', 0.0)
    except:
        p_pos, p_neg, p_neu = 0.0, 0.0, 0.0
    return p_pos, p_neg, p_neu

print("✅ Modèle sentiment chargé !")

In [ ]:
import re

# -------- Feature 1 : Intensifier Score --------
intensifiers = ['جدا', 'جداً', 'برشة', 'ياسر', 'للغاية', 'أكيد', 'كثير', 'بزاف']

def intensifier_score(text):
    count = sum(1 for word in intensifiers if word in text)
    return count

# -------- Feature 2 : Irony Marker Score --------
irony_markers = ['طبعاً', 'طبعا', 'أكيد', 'يا سلام', 'ممتاز', 'رائع',
                 'شكراً', 'شكرا', 'بالتوفيق', 'يعطيك العافية']

def irony_score(text):
    count = sum(1 for marker in irony_markers if marker in text)
    return count

# -------- Feature 3 : Negated Positive Sentiment --------
negations = ['موش', 'مش', 'ليس', 'لا', 'ما', 'مو']
positive_words = ['باهي', 'رائع', 'ممتاز', 'حلو', 'زين', 'كويس', 'جميل']

def negated_positive(text):
    words = text.split()
    for i, word in enumerate(words[:-1]):
        if word in negations and words[i+1] in positive_words:
            return 1
    return 0

# -------- Feature 4 : Negated Negative Sentiment --------
negative_words = ['خايب', 'سيئ', 'وحش', 'خسارة', 'فاشل', 'بايخ']

def negated_negative(text):
    words = text.split()
    for i, word in enumerate(words[:-1]):
        if word in negations and words[i+1] in negative_words:
            return 1
    return 0

# -------- Feature 5 : Hyperbole Score --------
hyperboles = ['أفضل شيء', 'في الكون', 'مليون مرة', 'أموت من', 'في العالم',
              'في التاريخ', 'ما شفت', 'ما في أحسن']

def hyperbole_score(text):
    count = sum(1 for h in hyperboles if h in text)
    return count

# -------- Feature 6 : Exclamation Density --------
def exclamation_density(text):
    count = text.count('!') + text.count('！')
    return min(count / (len(text) + 1), 1.0)

# -------- Feature 7 : Question Density --------
def question_density(text):
    count = text.count('?') + text.count('؟')
    return min(count / (len(text) + 1), 1.0)

# -------- Feature 8 : Latin Ratio --------
def latin_ratio(text):
    latin_chars = len(re.findall(r'[a-zA-Z]', text))
    return latin_chars / (len(text) + 1)

print("✅ Fonctions des 8 features définies !")

In [ ]:
def extract_all_features(text):
    # Sentiment (3 features)
    p_pos, p_neg, p_neu = get_sentiment_features(text)
    
    # Lexical (8 features)
    f1 = intensifier_score(text)
    f2 = irony_score(text)
    f3 = negated_positive(text)
    f4 = negated_negative(text)
    f5 = hyperbole_score(text)
    f6 = exclamation_density(text)
    f7 = question_density(text)
    f8 = latin_ratio(text)
    
    return [p_pos, p_neg, p_neu, f1, f2, f3, f4, f5, f6, f7, f8]

# Colonnes
columns = ['p_pos', 'p_neg', 'p_neu',
           'intensifier', 'irony', 'neg_pos', 'neg_neg',
           'hyperbole', 'exclamation', 'question', 'latin_ratio']

print("⏳ Extraction features train...")
train_features = df_train["tweet"].apply(extract_all_features)
train_prag_df = pd.DataFrame(train_features.tolist(), columns=columns)

print("⏳ Extraction features test...")
test_features = df_test["tweet"].apply(extract_all_features)
test_prag_df = pd.DataFrame(test_features.tolist(), columns=columns)

print("✅ Features extraites !")
print(train_prag_df.head())

In [ ]:
print(f"Train : {len(df_train)} tweets")
print(f"Test : {len(df_test)} tweets")
print(f"Total : {len(df_train) + len(df_test)} tweets")

In [ ]:
train_prag_df["label"] = df_train["sarcasm"].values
test_prag_df["label"] = df_test["sarcasm"].values

train_prag_df.to_csv("../outputs/train_pragmatic.csv", index=False)
test_prag_df.to_csv("../outputs/test_pragmatic.csv", index=False)

print("✅ Features pragmatiques sauvegardées !")
print(f"Train : {train_prag_df.shape}")
print(f"Test : {test_prag_df.shape}")

In [ ]:
print(df_train.columns)
print(df_train.head())

In [ ]:
train_prag_df["label"] = df_train["sarcasm"].values
test_prag_df["label"] = df_test["sarcasm"].values

train_prag_df.to_csv("../outputs/train_pragmatic.csv", index=False)
test_prag_df.to_csv("../outputs/test_pragmatic.csv", index=False)

print("✅ Features pragmatiques sauvegardées !")
print(f"Train : {train_prag_df.shape}")
print(f"Test : {test_prag_df.shape}")

In [ ]:
import pandas as pd

df_train = pd.read_csv("../data/training_data.csv")
df_test = pd.read_csv("../data/testing_data.csv")

print("=== Distribution du dataset Train ===")
print(df_train["sarcasm"].value_counts())
print(df_train["sarcasm"].value_counts(normalize=True))

print("\n=== Distribution du dataset Test ===")
print(df_test["sarcasm"].value_counts())
print(df_test["sarcasm"].value_counts(normalize=True))

In [ ]:
#Enrichir les features pragmatiques

In [ ]:
#Vocabulaire extrait du training set

In [ ]:
#1 Charger le dataset

In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter

# Charger le dataset
df_train = pd.read_csv("../data/training_data.csv")
df_test = pd.read_csv("../data/testing_data.csv")

print("✅ Dataset chargé !")
print(df_train.columns)
print(df_train.shape)

In [ ]:
#2 Extraire le vocabulaire discriminant

In [ ]:
# Séparer tweets sarcastiques et non sarcastiques
tweets_sarc = df_train[df_train["sarcasm"] == True]["tweet"].tolist()
tweets_non_sarc = df_train[df_train["sarcasm"] == False]["tweet"].tolist()

def tokenize_arabic(text):
    if not isinstance(text, str):
        return []
    # Garder seulement les mots arabes
    words = re.findall(r'[\u0600-\u06FF]+', text)
    return [w for w in words if len(w) > 2]  # mots de plus de 2 lettres

# Compter les mots dans chaque classe
counter_sarc = Counter()
counter_non_sarc = Counter()

for tweet in tweets_sarc:
    counter_sarc.update(tokenize_arabic(tweet))

for tweet in tweets_non_sarc:
    counter_non_sarc.update(tokenize_arabic(tweet))

print(f"Vocabulaire sarcastique : {len(counter_sarc)} mots uniques")
print(f"Vocabulaire non sarcastique : {len(counter_non_sarc)} mots uniques")

In [ ]:
#3  Calculer le score discriminant

In [ ]:
# Calculer le ratio : fréquence dans sarcasme / fréquence dans non-sarcasme
total_sarc = sum(counter_sarc.values())
total_non_sarc = sum(counter_non_sarc.values())

discriminant_scores = {}

for word in counter_sarc:
    freq_sarc = counter_sarc[word] / total_sarc
    freq_non_sarc = (counter_non_sarc.get(word, 0) + 1) / total_non_sarc
    score = freq_sarc / freq_non_sarc
    discriminant_scores[word] = score

# Trier par score discriminant décroissant
vocab_discriminant = sorted(
    discriminant_scores.items(),
    key=lambda x: x[1],
    reverse=True
)

# Garder les 100 mots les plus discriminants
TOP_N = 100
vocab_sarc = [word for word, score in vocab_discriminant[:TOP_N]]

print(f"✅ Top {TOP_N} mots discriminants extraits !")
print("\nExemples des 50 premiers mots :")
for word, score in vocab_discriminant[:50]:
    print(f"  {word} : score = {score:.3f}")

In [ ]:
#4  Nouvelle feature : Discriminant Vocab Score

In [ ]:
def discriminant_vocab_score(text):
    if not isinstance(text, str):
        return 0
    words = tokenize_arabic(text)
    if len(words) == 0:
        return 0
    count = sum(1 for w in words if w in vocab_sarc)
    return count / len(words)  # normalisé par la longueur

# Test rapide
exemple = "طبعاً كل شيء رائع ومميز في هذا البلد"
print(f"Score exemple : {discriminant_vocab_score(exemple):.4f}")
print("✅ Fonction discriminant_vocab_score définie !")

In [ ]:
#5 Nouvelles features stylistiques

In [ ]:
import re
import emoji

# ─── Feature 1 : Hashtag Count ───
def hashtag_count(text):
    if not isinstance(text, str):
        return 0
    return len(re.findall(r'#\w+', text))

# ─── Feature 2 : Emoji Count ───
def emoji_count(text):
    if not isinstance(text, str):
        return 0
    return sum(1 for c in text if c in emoji.EMOJI_DATA)

# ─── Feature 3 : Interjection Score ───
interjections = [
    'آه', 'أوه', 'يا إلهي', 'واو', 'يا ربي',
    'هههه', 'هاها', 'لول', 'lol', 'هيه',
    'آآآ', 'أف', 'اوف', 'يخي', 'يزي',
    'يلعن', 'تبا', 'لعنة', 'والله'
]

def interjection_score(text):
    if not isinstance(text, str):
        return 0
    text_lower = text.lower()
    return sum(1 for w in interjections if w in text_lower)

# ─── Feature 4 : Quotation Mark Score ───
def quotation_score(text):
    if not isinstance(text, str):
        return 0
    count = (text.count('"') + text.count("'") +
             text.count('«') + text.count('»') +
             text.count('\u201c') + text.count('\u201d'))
    return min(count / (len(text) + 1), 1.0)

# ─── Feature 5 : Repeated Characters Score ───
def repeated_chars_score(text):
    if not isinstance(text, str):
        return 0
    matches = re.findall(r'(.)\1{2,}', text)
    return min(len(matches) / (len(text) + 1), 1.0)

# ─── Feature 6 : Tweet Length ───
def tweet_length(text):
    if not isinstance(text, str):
        return 0
    return min(len(text) / 280, 1.0)

print("✅ 6 nouvelles features définies !")

test = "هههههه يا حكومة!! كلب واحد يسوى أكثر منكم 😂😂 #سياسة"
print(f"\nTest sur : {test}")
print(f"Hashtag Count      : {hashtag_count(test)}")
print(f"Emoji Count        : {emoji_count(test)}")
print(f"Interjection Score : {interjection_score(test)}")
print(f"Quotation Score    : {quotation_score(test):.4f}")
print(f"Repeated Chars     : {repeated_chars_score(test):.4f}")
print(f"Tweet Length       : {tweet_length(test):.4f}")
print(f"Discriminant Vocab : {discriminant_vocab_score(test):.4f}")

In [ ]:
#6 Fonction complète (11 + 7 = 18 features)

In [ ]:
from tqdm import tqdm
tqdm.pandas()

def extract_all_features_v2(text):
    if not isinstance(text, str):
        text = ""
    
    # ── Sentiment Proxy (3 features) ──
    p_pos, p_neg, p_neu = get_sentiment_features(text)
    
    # ── Lexical-Pragmatic originales (8 features) ──
    f1 = intensifier_score(text)
    f2 = irony_score(text)
    f3 = negated_positive(text)
    f4 = negated_negative(text)
    f5 = hyperbole_score(text)
    f6 = exclamation_density(text)
    f7 = question_density(text)
    f8 = latin_ratio(text)
    
    # ── Nouvelles features (7 features) ──
    f9  = discriminant_vocab_score(text)
    f10 = hashtag_count(text)
    f11 = emoji_count(text)
    f12 = interjection_score(text)
    f13 = quotation_score(text)
    f14 = repeated_chars_score(text)
    f15 = tweet_length(text)
    
    return [p_pos, p_neg, p_neu,
            f1, f2, f3, f4, f5, f6, f7, f8,
            f9, f10, f11, f12, f13, f14, f15]

columns_v2 = [
    'p_pos', 'p_neg', 'p_neu',
    'intensifier', 'irony', 'neg_pos', 'neg_neg',
    'hyperbole', 'exclamation', 'question', 'latin_ratio',
    'discriminant_vocab', 'hashtag_count', 'emoji_count',
    'interjection', 'quotation', 'repeated_chars', 'tweet_length'
]

print("⏳ Extraction des 18 features sur train...")
train_features_v2 = df_train["tweet"].progress_apply(extract_all_features_v2)
train_prag_v2 = pd.DataFrame(train_features_v2.tolist(), columns=columns_v2)

print("⏳ Extraction des 18 features sur test...")
test_features_v2 = df_test["tweet"].progress_apply(extract_all_features_v2)
test_prag_v2 = pd.DataFrame(test_features_v2.tolist(), columns=columns_v2)

print("✅ Extraction terminée !")
print(train_prag_v2.head())

In [ ]:
train_prag_v2["label"] = df_train["sarcasm"].values
test_prag_v2["label"] = df_test["sarcasm"].values

train_prag_v2.to_csv("../outputs/train_pragmatic_v2.csv", index=False)
test_prag_v2.to_csv("../outputs/test_pragmatic_v2.csv", index=False)

print("✅ Features v2 sauvegardées !")
print(f"Train : {train_prag_v2.shape}")
print(f"Test  : {test_prag_v2.shape}")
print(f"\nColonnes : {list(train_prag_v2.columns)}")